# GroupDNA - WhatsApp Chat Analyzer

## Project Setup

In [154]:
import numpy as np
from datetime import datetime, timedelta

file_name = "/content/hostel_bois.txt"

# Feature 1 - Chat Parser

In [155]:
def check_date(line):
    if len(line) < 8:
        return False

    if line[2] == "/" and line[5] == "/":
        return True

    return False

In [156]:
messages = []

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

for line in lines:

    line = line.strip()

    if line == "":
        continue

    if check_date(line):

        parts = line.split(" - ", 1)

        if len(parts) == 2:

            time = parts[0]
            text = parts[1]

            parts2 = text.split(": ", 1)

            if len(parts2) == 2:

                name = parts2[0]
                message = parts2[1]

                messages.append(
                    [time, name, message]
                )

    else:

        if len(messages) > 0:
            messages[-1][2] += " " + line

In [157]:
for message in messages:

    message[0] = datetime.strptime(
        message[0],
        "%d/%m/%y, %H:%M"
    )

print("Total messages:", len(messages))

print("\nFirst 5 messages")

for message in messages[:5]:
    print(message)

Total messages: 3174

First 5 messages
[datetime.datetime(2024, 4, 1, 1, 17), 'Rahul', 'scene fix']
[datetime.datetime(2024, 4, 1, 1, 17), 'Rahul', 'haan']
[datetime.datetime(2024, 4, 1, 1, 18), 'Rahul', 'kya scene']
[datetime.datetime(2024, 4, 1, 2, 13), 'Rahul', 'abhi free hai?']
[datetime.datetime(2024, 4, 1, 2, 13), 'Rahul', 'abey']


# Feature 2 - Group Overview

In [158]:
people = []

for message in messages:

    name = message[1]

    if name not in people:
        people.append(name)

message_count = {}

word_count = {}

for person in people:
    message_count[person] = 0
    word_count[person] = 0

for message in messages:

    person = message[1]
    text = message[2]

    if text != "<Media omitted>" and text != "This message was deleted":

        message_count[person] += 1

        words = text.split()

        word_count[person] += len(words)

In [159]:
first_date = messages[0][0]
last_date = messages[-1][0]

days = (
    last_date.date() - first_date.date()
).days + 1

total_messages = 0

for person in people:
    total_messages += message_count[person]

print("=" * 50)
print("              GROUP OVERVIEW")
print("=" * 50)

print("Participants:", len(people))
print("Total messages:", total_messages)
print("Number of days:", days)

print(
    "Start date:",
    first_date.strftime("%d-%m-%Y")
)

print(
    "End date:",
    last_date.strftime("%d-%m-%Y")
)

print("\nMessages per person")

for person in people:

    count = message_count[person]

    percentage = (
        count / total_messages
    ) * 100

    average_words = (
        word_count[person] / count
    )

    print(
        person,
        ":",
        count,
        "(",
        round(percentage, 1),
        "%)"
    )

    print(
        "Average words:",
        round(average_words, 1)
    )

              GROUP OVERVIEW
Participants: 6
Total messages: 3127
Number of days: 60
Start date: 01-04-2024
End date: 30-05-2024

Messages per person
Rahul : 940 ( 30.1 %)
Average words: 2.6
Priya : 712 ( 22.8 %)
Average words: 5.0
Karan : 345 ( 11.0 %)
Average words: 57.0
Neha : 624 ( 20.0 %)
Average words: 5.3
Aman : 484 ( 15.5 %)
Average words: 5.0
Vikas : 22 ( 0.7 %)
Average words: 1.8


# Feature 3 - Busiest Day and Hour

In [160]:
day_count = {}
hour_count = {}

for message in messages:

    text = message[2]

    if text == "<Media omitted>" or text == "This message was deleted":
        continue

    date = message[0].date()
    hour = message[0].hour

    if date not in day_count:
        day_count[date] = 0

    if hour not in hour_count:
        hour_count[hour] = 0

    day_count[date] += 1
    hour_count[hour] += 1

busy_day = max(
    day_count,
    key=day_count.get
)

busy_hour = max(
    hour_count,
    key=hour_count.get
)

print("Busiest day:", busy_day)
print("Messages:", day_count[busy_day])

print(
    "Busiest hour:",
    str(busy_hour) + ":00"
)

print(
    "Messages:",
    hour_count[busy_hour]
)

Busiest day: 2024-05-04
Messages: 74
Busiest hour: 18:00
Messages: 244


# Feature 4 - Activity Heatmap

In [161]:
activity = np.zeros(
    (len(people), 24),
    dtype=int
)

for message in messages:

    text = message[2]

    if text == "<Media omitted>" or text == "This message was deleted":
        continue

    person = message[1]
    hour = message[0].hour

    row = people.index(person)

    activity[row][hour] += 1

print("Activity matrix")
print(activity)

Activity matrix
[[  3  15  17  17  22   9  17  17  23  17  25  15  57  48  45  53  71  48
  102  74  40  92  60  53]
 [  0   0   0   0   0   0  13  20  46  64  61  61  57  48  44  28  32  40
   38  59  42  32  18   9]
 [  0   0   0   0   0   0   0   4  11  16  19  16  36  22  32  26  27  27
   24  32  23  14   9   7]
 [  0   0   0   0   0  19   3  13  35  51  52  21  39  36  26   9  36  47
   62  49  44  26  26  30]
 [ 53  67  66  60  87   0   0   0   0   0   0   0   0   0  14  11  18   5
   16   8  12  11   0  56]
 [  0   0   0   0   0   0   0   1   2   1   1   0   1   2   0   1   1   3
    2   2   1   1   1   2]]


In [162]:
print("ACTIVITY HEATMAP")

print("Person   ", end="")

for hour in range(24):
    print(f"{hour:02}", end=" ")

print()

for i in range(len(people)):

    print(f"{people[i]:8}", end=" ")

    for hour in range(24):

        value = activity[i][hour]

        if value == 0:
            print(".", end=" ")

        elif value < 5:
            print("░", end=" ")

        elif value < 10:
            print("▒", end=" ")

        else:
            print("█", end=" ")

    print()

ACTIVITY HEATMAP
Person   00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul    ░ █ █ █ █ ▒ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ 
Priya    . . . . . . █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ █ ▒ 
Karan    . . . . . . . ░ █ █ █ █ █ █ █ █ █ █ █ █ █ █ ▒ ▒ 
Neha     . . . . . █ ░ █ █ █ █ █ █ █ █ ▒ █ █ █ █ █ █ █ █ 
Aman     █ █ █ █ █ . . . . . . . . . █ █ █ ▒ █ ▒ █ █ . █ 
Vikas    . . . . . . . ░ ░ ░ ░ . ░ ░ . ░ ░ ░ ░ ░ ░ ░ ░ ░ 


# Feature 5 - Favourite Words

In [163]:
stop_words = [
    "the", "is", "a", "an", "and", "or", "to", "of", "in", "on", "for",
    "i", "you", "me", "my", "we", "are", "was", "this", "how", "so",
    "about", "am", "today", "at", "he", "his", "which", "im", "cant",
    "with", "it", "that", "have", "be", "do", "not", "your", "all"
]

words_count = {}
for message in messages:
    text = message[2].lower()
    if text == "":
        continue
    for symbol in ".,!?;:\"'()[]":
        text = text.replace(symbol, "")
    words = text.split()
    for word in words:
        if word not in stop_words:
            if word not in words_count:
                words_count[word] = 0
            words_count[word] += 1

In [164]:
top_words = sorted(
    words_count.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 10 WORDS")

for word, count in top_words[:10]:

    print(
        word,
        ":",
        count
    )

TOP 10 WORDS
guys : 318
hai : 268
just : 208
everyone : 187
telling : 179
from : 174
up : 172
bhai : 160
one : 157
had : 151


In [165]:
person_words = {}

for person in people:
    person_words[person] = {}

for message in messages:

    person = message[1]
    text = message[2].lower()

    if text == "<media omitted>":
        continue

    for symbol in ".,!?;:\"'()[]":

        text = text.replace(
            symbol,
            ""
        )

    words = text.split()

    for word in words:

        if word not in stop_words:

            if word not in person_words[person]:
                person_words[person][word] = 0

            person_words[person][word] += 1


print("TOP WORDS PER PERSON")

for person in people:

    words = person_words[person]

    top = sorted(
        words.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print("\n", person)

    for word, count in top[:5]:

        print(
            word,
            ":",
            count
        )

TOP WORDS PER PERSON

 Rahul
hai : 263
bhai : 159
scene : 144
kya : 133
yaar : 105

 Priya
please : 141
everyone : 137
aman : 93
anyone : 90
okay : 80

 Karan
telling : 179
up : 164
just : 155
from : 154
started : 150

 Neha
guys : 101
ok : 52
what : 48
no : 48
way : 48

 Aman
sleep : 71
anyone : 49
3 : 45
wonder : 43
night : 41

 Vikas
hai : 5
haha : 4
sorry : 3
busy : 3
tha : 3


# Feature 6 - Response Speed and Silent Streaks

In [166]:
response_times = {}

for person in people:
    response_times[person] = []

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    previous_person = previous[1]
    current_person = current[1]

    if previous_person != current_person:

        gap = (
            current[0] - previous[0]
        )

        seconds = gap.total_seconds()

        if seconds >= 0:

            response_times[
                current_person
            ].append(seconds)

In [167]:
average_response = {}

for person in people:

    times = response_times[person]

    if len(times) > 0:

        average_response[person] = (
            sum(times) / len(times)
        )

    else:

        average_response[person] = 0


def show_time(seconds):

    if seconds < 60:
        return str(round(seconds, 1)) + " seconds"

    minutes = seconds / 60

    if minutes < 60:
        return str(round(minutes, 1)) + " minutes"

    hours = minutes / 60

    return str(round(hours, 1)) + " hours"


fastest = min(
    average_response,
    key=average_response.get
)

slowest = max(
    average_response,
    key=average_response.get
)

print(
    "Fastest replier:",
    fastest,
    show_time(average_response[fastest])
)

print(
    "Slowest replier:",
    slowest,
    show_time(average_response[slowest])
)

Fastest replier: Rahul 34.9 minutes
Slowest replier: Aman 55.4 minutes


In [168]:
active_dates = {}
for person in people:
    active_dates[person] = set()

for message in messages:
    person = message[1]
    date = message[0].date()
    active_dates[person].add(date)

silent_streaks = {}
for person in people:
    current = first_date.date()
    longest = 0
    streak = 0
    while current <= last_date.date():
        if current not in active_dates[person]:
            streak += 1
            if streak > longest:
                longest = streak
        else:
            streak = 0
        current += timedelta(days=1)
    silent_streaks[person] = longest

print("LONGEST SILENT STREAKS")
for person in people:
    print(person, ":", silent_streaks[person], "days")

LONGEST SILENT STREAKS
Rahul : 0 days
Priya : 0 days
Karan : 0 days
Neha : 0 days
Aman : 0 days
Vikas : 11 days


# Feature 7 - Personality Archetypes

In [169]:
archetype_scores = {}

caring_words = ["okay", "safe", "eat", "sleep", "take care", "are you", "please", "reminder", "drink water", "don't forget"]
funny_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

for person in people:
    archetype_scores[person] = {}
    person_msgs = [m for m in messages if m[1] == person and m[2] != "" and m[2] != "This message was deleted"]
    total = len(person_msgs)

    if total == 0:
        continue

    # Spammer Burst Calculation
    burst_total = 0
    burst_count = 0
    current_burst = 0
    for i in range(len(messages)):
        if messages[i][1] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                burst_total += current_burst
                burst_count += 1
                current_burst = 0
    if current_burst > 0:
        burst_total += current_burst
        burst_count += 1
    spammer_score = (burst_total / burst_count) if burst_count > 0 else 0

    # Message Metrics
    caring_count = 0
    night_count = 0
    total_words = 0
    drama_count = 0
    funny_count = 0
    question_count = 0

    for msg in person_msgs:
        text = msg[2]
        lower_text = text.lower()
        total_words += len(text.split())

        for word in caring_words:
            if word in lower_text:
                caring_count += 1

        if msg[0].hour >= 23 or msg[0].hour <= 4:
            night_count += 1

        if (len(text) >= 3 and text.isupper()) or text.count("!") >= 2:
            drama_count += 1

        for word in funny_words:
            if word in lower_text:
                funny_count += 1

        if text.strip().endswith("?"):
            question_count += 1

    # Scaled Scores
    archetype_scores[person]["THE SPAMMER"] = spammer_score if spammer_score > 3 else 0
    archetype_scores[person]["THE GROUP MOM"] = (caring_count / total) if (caring_count / total) > 0.30 else 0
    archetype_scores[person]["THE NIGHT OWL"] = (night_count / total) if (night_count / total) > 0.60 else 0
    archetype_scores[person]["THE STORYTELLER"] = (total_words / total) if (total_words / total) > 30 else 0
    archetype_scores[person]["THE DRAMA QUEEN"] = (drama_count / total) if (drama_count / total) > 0.30 else 0
    archetype_scores[person]["THE GHOST"] = (silent_streaks[person] / days) if (silent_streaks[person] / days) > 0.60 else 0
    archetype_scores[person]["THE COMEDIAN"] = funny_count / total
    archetype_scores[person]["THE QUESTION MASTER"] = (question_count / total) if (question_count / total) > 0.25 else 0

In [170]:
personality = {}

for person in people:

    best_archetype = ""
    best_score = -1

    for archetype in archetype_scores[person]:

        score = archetype_scores[person][archetype]

        if score > best_score:

            best_score = score
            best_archetype = archetype

    personality[person] = best_archetype

print("PERSONALITY ARCHETYPES")

for person in people:

    print(
        person,
        "->",
        personality[person]
    )

PERSONALITY ARCHETYPES
Rahul -> THE SPAMMER
Priya -> THE GROUP MOM
Karan -> THE STORYTELLER
Neha -> THE DRAMA QUEEN
Aman -> THE NIGHT OWL
Vikas -> THE COMEDIAN


In [171]:
print("\nARCHETYPE SCORES")

for person in people:

    print("\n" + person)

    for archetype in archetype_scores[person]:

        score = archetype_scores[person][archetype]

        print(
            archetype,
            ":",
            round(score, 3)
        )


ARCHETYPE SCORES

Rahul
THE SPAMMER : 4.517
THE GROUP MOM : 0
THE NIGHT OWL : 0
THE STORYTELLER : 0
THE DRAMA QUEEN : 0
THE GHOST : 0
THE COMEDIAN : 0.032
THE QUESTION MASTER : 0

Priya
THE SPAMMER : 0
THE GROUP MOM : 0.867
THE NIGHT OWL : 0
THE STORYTELLER : 0
THE DRAMA QUEEN : 0
THE GHOST : 0
THE COMEDIAN : 0.0
THE QUESTION MASTER : 0.293

Karan
THE SPAMMER : 0
THE GROUP MOM : 0
THE NIGHT OWL : 0
THE STORYTELLER : 55.952
THE DRAMA QUEEN : 0
THE GHOST : 0
THE COMEDIAN : 0.0
THE QUESTION MASTER : 0

Neha
THE SPAMMER : 0
THE GROUP MOM : 0
THE NIGHT OWL : 0
THE STORYTELLER : 0
THE DRAMA QUEEN : 0.625
THE GHOST : 0
THE COMEDIAN : 0.0
THE QUESTION MASTER : 0

Aman
THE SPAMMER : 0
THE GROUP MOM : 0
THE NIGHT OWL : 0.797
THE STORYTELLER : 0
THE DRAMA QUEEN : 0
THE GHOST : 0
THE COMEDIAN : 0.0
THE QUESTION MASTER : 0

Vikas
THE SPAMMER : 0
THE GROUP MOM : 0
THE NIGHT OWL : 0
THE STORYTELLER : 0
THE DRAMA QUEEN : 0
THE GHOST : 0
THE COMEDIAN : 0.167
THE QUESTION MASTER : 0


# Feature 8 - Final GroupDNA Report

In [172]:
print("=" * 60)
print("GROUPDNA REPORT")
print("=" * 60)

print(
    "Period:",
    first_date.strftime("%d-%m-%Y"),
    "to",
    last_date.strftime("%d-%m-%Y")
)

print("Days:", days)

print("Messages:", total_messages)

print("Participants:", len(people))

print("\nMESSAGES PER PERSON")

for person in people:

    count = message_count[person]

    percentage = (
        count / total_messages
    ) * 100

    print(
        person,
        ":",
        count,
        "(",
        round(percentage, 1),
        "%)"
    )

print("\nBUSIEST DAY")

print(
    busy_day,
    ":",
    day_count[busy_day],
    "messages"
)

print("\nBUSIEST HOUR")

print(
    str(busy_hour) + ":00",
    ":",
    hour_count[busy_hour],
    "messages"
)

print("\nTOP WORDS")

for word, count in top_words[:10]:

    print(
        word,
        ":",
        count
    )

print("\nPERSONALITY ARCHETYPES")

for person in people:

    print(
        person,
        "->",
        personality[person]
    )

GROUPDNA REPORT
Period: 01-04-2024 to 30-05-2024
Days: 60
Messages: 3127
Participants: 6

MESSAGES PER PERSON
Rahul : 940 ( 30.1 %)
Priya : 712 ( 22.8 %)
Karan : 345 ( 11.0 %)
Neha : 624 ( 20.0 %)
Aman : 484 ( 15.5 %)
Vikas : 22 ( 0.7 %)

BUSIEST DAY
2024-05-04 : 74 messages

BUSIEST HOUR
18:00 : 244 messages

TOP WORDS
guys : 318
hai : 268
just : 208
everyone : 187
telling : 179
from : 174
up : 172
bhai : 160
one : 157
had : 151

PERSONALITY ARCHETYPES
Rahul -> THE SPAMMER
Priya -> THE GROUP MOM
Karan -> THE STORYTELLER
Neha -> THE DRAMA QUEEN
Aman -> THE NIGHT OWL
Vikas -> THE COMEDIAN


# Reflection

The hardest part of this project
was reading and understanding the WhatsApp chat data.

I learned how to read a text file and use Python lists, dictionaries, loops, functions, datetime and NumPy.

I liked the activity heatmap because it showed when the group was most active.

I also found the personality feature interesting because it showed different types of group members.

If I improve this project, I would add more features and make it work with different WhatsApp chat formats.

This project helped me learn more about Python and basic data analysis.